# 20 — Report readiness check

Fail loudly if the report bundle is missing core evidence, if only provisional seed trade exists, or if the price hypothesis has not been analysed.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

import json


In [ ]:
manifest_path = PATHS.report_inputs / "report_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Run notebook 19 first.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
checks = []
checks.append({"check": "core_report_bundle_complete", "passed": bool(manifest.get("complete")), "detail": str(manifest.get("missing", []))})
trade = pd.read_csv(PATHS.processed / "fuel_trade_annual.csv")
seed_only = "status" in trade.columns and set(trade["status"].dropna()) == {"seed_provisional"}
checks.append({"check": "trade_not_seed_only", "passed": not seed_only, "detail": "Re-run JODI/DGEG acquisition before publication" if seed_only else "downloaded/cross-checked trade available"})
jodi_completeness_files = [PATHS.metrics / name for name in ["jodi_trade_annual_completeness.csv", "jodi_demand_annual_completeness.csv", "jodi_refinery_output_annual_completeness.csv"]]
checks.append({"check": "jodi_annual_completeness_documented", "passed": all(path.exists() for path in jodi_completeness_files), "detail": "Annual JODI totals must document n_months, missing_months and assessment_status"})
checks.append({"check": "monthly_event_panel_available", "passed": (PATHS.processed / "fuel_monthly_analytical_panel.csv").exists(), "detail": "Required to separate May 2021 closure timing from the March 2022 energy shock"})
checks.append({"check": "eurostat_balance_available", "passed": (PATHS.processed / "eurostat_physical_balance_panel.csv").exists(), "detail": "Required before full petroleum-product balance claims"})
checks.append({"check": "price_comovement_models_available", "passed": (PATHS.metrics / "price_comovement_models.csv").exists(), "detail": "Price-exposure claim must remain untested if false"})
checks.append({"check": "price_stationarity_checked", "passed": (PATHS.metrics / "price_stationarity_diagnostics.csv").exists(), "detail": "Required before interpreting weekly price-level regressions"})
checks.append({"check": "dgeg_trade_reconciled", "passed": (PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv").exists(), "detail": "Required before physical import-dependence claims"})
readiness = pd.DataFrame(checks)
persist_dataframe(readiness, PATHS.metrics / "report_readiness.csv", key_columns=["check"])
display(readiness)
publication_blockers = ["core_report_bundle_complete", "trade_not_seed_only", "jodi_annual_completeness_documented", "monthly_event_panel_available", "eurostat_balance_available", "dgeg_trade_reconciled"]
if not readiness.loc[readiness["check"].isin(publication_blockers), "passed"].all():
    raise RuntimeError("Core publication-readiness checks failed. See data/metrics/report_readiness.csv")


A failed price co-movement check does not block a purely physical-supply report, but it **does** block any conclusion that refining reconfiguration changed domestic price exposure. DGEG reconciliation, monthly event timing and JODI annual-completeness diagnostics are publication blockers for physical import-dependence claims.
